In [71]:
import json
import os
import  numpy as np
import pandas as pd
from tensorflow.keras import Model
from tensorflow.keras.utils import Sequence
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten

In [72]:
test_img="./arcade/stenosis/test/images/"
train_img="./arcade/stenosis/train/images/"
val_img="./arcade/stenosis/val/images/"

In [73]:
js_train="./arcade/stenosis/train/annotations/train.json"
js_val="./arcade/stenosis/val/annotations/val.json"
js_test="./arcade/stenosis/test/annotations/test.json"

with open (js_train,"r") as f:
    js_tra=json.load(f)

with open (js_val,"r") as b:
    js_v=json.load(b)

with open (js_test,"r") as c:
    js_te=json.load(c)

In [74]:
print(f" count train:{len(os.listdir(train_img))}")
print(f" count val:{len(os.listdir(val_img))}")
print(f" count test:{len(os.listdir(test_img))}")

 count train:1000
 count val:200
 count test:300


In [75]:
print("تعداد تصاویر:", len(js_tra['images']))
print("تعداد annotations:", len(js_tra['annotations']))
print("تعداد categories:", len(js_tra['categories']))

تعداد تصاویر: 1000
تعداد annotations: 1625
تعداد categories: 26


In [76]:
import json
import os
import shutil
from pathlib import Path

BASE_DIR = "./arcade/stenosis"
OUTPUT_DIR = "./yolo_dataset"

for split in ['train', 'val', 'test']:
    os.makedirs(f'{OUTPUT_DIR}/images/{split}', exist_ok=True)
    os.makedirs(f'{OUTPUT_DIR}/labels/{split}', exist_ok=True)

for split in ['train', 'val', 'test']:
    json_path=f"{BASE_DIR}/{split}/annotations/{split}.json"
    images_dir=f"{BASE_DIR}/{split}/images"

    print(f"\n Processing {split}...")

    with open(json_path, 'r') as f:
        data = json.load(f)

    image_dict = {img['id']: img for img in data['images']}

    count = 0
    for ann in data['annotations']:
        img_id = ann['image_id']
        img_info = image_dict[img_id]

        width = img_info['width']
        height = img_info['height']
        file_name = img_info['file_name']

        segmentation = ann['segmentation'][0]
        seg_normalized = []
        for i in range(0, len(segmentation), 2):
            x = segmentation[i] / width
            y = segmentation[i+1] / height
            seg_normalized.extend([x, y])

        label_file = f"{OUTPUT_DIR}/labels/{split}/{Path(file_name).stem}.txt"
        with open(label_file, 'a') as f:
            category_id = ann['category_id'] - 1
            seg_str = ' '.join([f"{coord:.6f}" for coord in seg_normalized])
            f.write(f"{category_id} {seg_str}\n")

        src_img = f"{images_dir}/{file_name}"
        dst_img = f"{OUTPUT_DIR}/images/{split}/{file_name}"
        if os.path.exists(src_img) and not os.path.exists(dst_img):
            shutil.copy(src_img, dst_img)
            count += 1

    print(f" {split}: {count} images copied")

yaml_content = f"""path: {os.path.abspath(OUTPUT_DIR)}
train: images/train
val: images/val
test: images/test

nc: 26
names: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '9a',
        '10', '10a', '11', '12', '12a', '13', '14', '14a',
        '15', '16', '16a', '16b', '16c', '12b', '14b', 'stenosis']
"""

with open('dataset.yaml', 'w', encoding='utf-8') as f:
    f.write(yaml_content)

print("\n Conversion completed!")
print(" dataset.yaml file created")
print(f" {OUTPUT_DIR} folder is ready to use with YOLO")


 Processing train...
 train: 0 images copied

 Processing val...
 val: 0 images copied

 Processing test...
 test: 0 images copied

 Conversion completed!
 dataset.yaml file created
 ./yolo_dataset folder is ready to use with YOLO


In [77]:
from ultralytics import YOLO
yolo_data=YOLO("yolo_dataset")

WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
